# SpamShield AI - SMS Spam EDA

Exploratory data analysis on the SMS Spam Collection dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q wordcloud transformers pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Dataset

In [ ]:
DATA_PATH = '/content/drive/My Drive/data/smsspamcollection/SMSSpamCollection'

df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['label', 'text'], encoding='latin-1')
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

## 2. Class Distribution

In [ ]:
df['label_name'] = df['label'].map({0: 'Ham', 1: 'Spam'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['label_name'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

df['label_name'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=axes[1], colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Class Proportion')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Text Length Analysis

In [ ]:
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for label, color, name in [(0, '#2ecc71', 'Ham'), (1, '#e74c3c', 'Spam')]:
    subset = df[df['label'] == label]
    axes[0, 0].hist(subset['text_length'], bins=50, alpha=0.6, color=color, label=name, density=True)
    axes[0, 1].hist(subset['word_count'], bins=50, alpha=0.6, color=color, label=name, density=True)

axes[0, 0].set_title('Text Length Distribution (chars)')
axes[0, 0].set_xlabel('Characters')
axes[0, 0].legend()

axes[0, 1].set_title('Word Count Distribution')
axes[0, 1].set_xlabel('Words')
axes[0, 1].legend()

df.boxplot(column='text_length', by='label_name', ax=axes[1, 0])
axes[1, 0].set_title('Text Length by Class')
axes[1, 0].set_xlabel('')

df.boxplot(column='word_count', by='label_name', ax=axes[1, 1])
axes[1, 1].set_title('Word Count by Class')
axes[1, 1].set_xlabel('')

plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Word Clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (label, color, title) in enumerate([(0, 'Greens', 'Ham'), (1, 'Reds', 'Spam')]):
    text = ' '.join(df[df['label'] == label]['text'])
    wc = WordCloud(width=800, height=400, background_color='white', colormap=color, max_words=200).generate(text)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'{title} Word Cloud')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 5. Token Statistics

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

df['token_count'] = df['text'].apply(lambda x: len(tokenizer.encode(str(x))))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [(0, '#2ecc71', 'Ham'), (1, '#e74c3c', 'Spam')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['token_count'], bins=50, alpha=0.6, color=color, label=name)

axes[0].set_title('Token Count Distribution (DistilBERT)')
axes[0].set_xlabel('Token Count')
axes[0].legend()

df.boxplot(column='token_count', by='label_name', ax=axes[1])
axes[1].set_title('Token Count by Class')
axes[1].set_xlabel('')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
print('Token Count Summary:')
print(df.groupby('label_name')['token_count'].describe().round(2))
print(f'\nTokens > 128: {(df["token_count"] > 128).sum()} ({(df["token_count"] > 128).mean()*100:.1f}%)')

## 6. Summary

In [ ]:
print('=' * 50)
print('DATASET SUMMARY')
print('=' * 50)
print(f'Total messages: {len(df):,}')
print(f'Ham: {(df["label"] == 0).sum():,} ({(df["label"] == 0).mean()*100:.1f}%)')
print(f'Spam: {(df["label"] == 1).sum():,} ({(df["label"] == 1).mean()*100:.1f}%)')
print(f'\nText Length (chars):')
print(df.groupby('label_name')['text_length'].describe().round(2))
print(f'\nWord Count:')
print(df.groupby('label_name')['word_count'].describe().round(2))

## 7. Save as Parquet

In [ ]:
OUTPUT_PATH = '/content/drive/My Drive/data/sms_spam.parquet'

save_df = df[['text', 'label']].copy()
save_df.to_parquet(OUTPUT_PATH, index=False)

print(f'Saved {len(save_df):,} rows to {OUTPUT_PATH}')
print(f'File size: {__import__("os").path.getsize(OUTPUT_PATH) / 1e6:.1f} MB')

# Verify
verify = pd.read_parquet(OUTPUT_PATH)
print(f'Verified: {verify.shape}, columns: {verify.columns.tolist()}')